In [2]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 645.7 kB/s eta 0:00:03
   -------------- ------------------------- 0.8/2.1 MB 958.5 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.1 MB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.1 MB/s eta 0:00:02
   ------------------------ --------------- 1.3/2.1 MB 871.6 kB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 871.6 kB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 871.6 kB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 871.6 kB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 665.7 kB/s eta 0:00:01
   -----------------------


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    f1_score, precision_recall_curve,
    confusion_matrix, classification_report
)
import optuna
from optuna.samplers import TPESampler

# --- Import des données ---
train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test = pd.read_csv('../6.Data/Yann_Process_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

selected_features = [col for col in train.columns if col not in [target, id_col]]

X = train[selected_features]
y = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scale = (y_train == 0).sum() / (y_train == 1).sum()

# --- Optuna ---
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight': scale,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print('Meilleurs paramètres:', study.best_params)
print(f'Meilleur F1 CV: {study.best_value:.4f}')

# --- Entraînement final ---
best_params = study.best_params
best_params.update({
    'scale_pos_weight': scale,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
})

best_model = LGBMClassifier(**best_params)
best_model.fit(X_train, y_train)

# --- Optimisation du seuil ---
y_proba = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[f1_scores.argmax()]

y_pred = (y_proba >= best_threshold).astype(int)
print(f"Seuil optimal : {best_threshold:.3f}")
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

Best trial: 96. Best value: 0.38032: 100%|██████████| 100/100 [32:17<00:00, 19.38s/it]


Meilleurs paramètres: {'n_estimators': 420, 'max_depth': 8, 'learning_rate': 0.06460023050326047, 'num_leaves': 133, 'subsample': 0.916090759832928, 'colsample_bytree': 0.7159609484779333, 'min_child_samples': 47, 'reg_alpha': 7.930651027071483e-08, 'reg_lambda': 5.998360260690286e-07}
Meilleur F1 CV: 0.3803
Seuil optimal : 0.634
[[27054  2191]
 [ 1574  1181]]
              precision    recall  f1-score   support

           0       0.95      0.93      0.93     29245
           1       0.35      0.43      0.39      2755

    accuracy                           0.88     32000
   macro avg       0.65      0.68      0.66     32000
weighted avg       0.89      0.88      0.89     32000

